# Step 4: the sensitivity batch — 120 ReEDS cases

**What this notebook does.** Step 3's 25 production runs (18 `smr100_{sched}_{p05|p50|p95}`
+ 6 `large100_{sched}_p50` comparators + the equality flip copy) are back and checked green
(2026-08-12, `z-ethan/step3_checks/`). This notebook generates the Step 4 batch —
`cases_nuclearlearning_step4.csv`, 120 run columns — from two ratified decisions
(2026-08-12):

1. **Market sensitivities, full coverage:** every smr100 percentile case × six sensitivities
   (`gaslo`/`gashi` gas price, `demhi` high demand, `relo`/`rehi` RE+storage cost,
   `translim` constrained transmission) = 108 columns. These need **no new nuclear input
   files**: each column points at its base case's plantchar/financials/construction-times
   files (the same file-pointer mechanism the Step 3 equality flip copy uses) and overrides
   only the sensitivity switches.
2. **Traditional-nuclear arm completed to percentiles:** `large100_{sched}_{p05|p95}` = 12
   columns, the drawn worlds at P5/P95 of the **pure-large** program-NPV ranking over the
   identical 10k worlds (the P50 cases already ran in Step 3 and are never re-run). These
   12 are the only cases needing new input files (24 plantchar + 12 financials + 12
   construction-times).

A third arm — 18 intertemporal (`timetype=int`) foresight columns — was ratified 2026-08-12
and **cancelled 2026-08-13**: NREL confirmed by email that the intertemporal solver has not
worked for years. The sequential/myopic caveat stays a caveat (issue 5's foresight-vs-myopia
wedge remains analytical, not empirical).

Run arithmetic: 25 (Step 3, spent) + 120 = **145 total; the run ceiling is amended
133 → 145** (163 with the int arm was ratified 2026-08-12, reduced 2026-08-13 on its
cancellation).

The MC re-run below is verbatim `smr100_case_export.ipynb` code (ported at build time by
`_build_step4_case_export.py` with content asserts): same worlds, same seed streams, QA-0
pinned to `mc_perdraw.npz`, base-case selections asserted identical to the frozen
`exports/smr100/` registrations. Step 3 input artifacts are **read-only** here — a snapshot
guard asserts they are byte-identical at the end.


In [1]:
import json
import os
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# --- Paths: anchored to this notebook's folder, never the launch directory ---
NB_DIR = Path.cwd() if Path.cwd().name == "mc" else Path("z-ethan/mc").resolve()
assert (NB_DIR / "pris_loader.py").exists(), f"notebook folder not found from {Path.cwd()}"
REPO_ROOT = NB_DIR.parent.parent               # the ReEDS-nuclear-learning fork
EXPORTS = NB_DIR / "exports" / "step4"
FIGURES = NB_DIR / "figures"
for d in (EXPORTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

import sys
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

# --- Reproducibility: one master seed, independent named streams. The world streams use
# the COMPANION notebook's names ("world/{schedule}") so the drawn worlds here are
# bit-identical to mc_cost_trajectories.ipynb's (QA-0 asserts this); streams unique to
# this notebook are prefixed "smr100/". ---
MASTER_SEED = 20260715

def rng_stream(name):
    """An independent, reproducible random generator tied to a label."""
    return np.random.default_rng(np.random.SeedSequence((MASTER_SEED, zlib.crc32(name.encode()))))

# --- Analysis switches ---
COPULA_SET = "moderate"                # within-tech lr <-> boak correlation (as companions)
RANKING_FUNCTIONAL = "discounted_schedule_weighted_npv"  # issue-8 default weights, NPV cost object (2026-08-03)
# NOTE: deliberately differs from mc_cost_trajectories' financed-CAPEX ranking - the mc_ and
# smr100 case families' percentile labels are NOT on a common scale (see README).
N_DRAWS = int(os.environ.get("STEP4_DRAWS", 10000))  # per schedule; env override for test runs

# --- Model frame (identical to the companion notebooks) ---
YEARS = np.arange(2024, 2051)
T = len(YEARS)
def yi(y):
    """Index of calendar year y in the YEARS grid."""
    return int(np.where(YEARS == y)[0][0])

ANCHOR = 2030
N_BOAK_UNITS = 2.0
OMEGA = 1.0 / 3.0
CES_RHO_GRID = np.array([-1.0, 0.0, 1.0])   # symmetric since 2026-08-06 (companion S1)
X_SPILL_MAX = 0.30     # cross-tech spillover fractions drawn U(0, X_SPILL_MAX) per direction
TECH = {
    "large": {"unit_gw": 1.000, "lr_lo": 0.03, "lr_hi": 0.12,
              "boak_lo": 5250.0, "boak_hi": 7750.0},
    "smr":   {"unit_gw": 0.300, "lr_lo": 0.03, "lr_hi": 0.16,
              "boak_lo": 5500.0, "boak_hi": 10000.0},
}
UNIT_FOREIGN_GW = 1.0
U_GRID = np.linspace(0.0, 1.0, 21)

print(f"notebook dir : {NB_DIR}")
print(f"switches     : copula={COPULA_SET}, ranking={RANKING_FUNCTIONAL}, draws/schedule={N_DRAWS}")
print("deployment   : 100% SMR (large rides the loser channel: intl spillover + x * SMR program)")

notebook dir : C:\Users\ethan\code\research\ReEDS-nuclear-learning\z-ethan\mc
switches     : copula=moderate, ranking=discounted_schedule_weighted_npv, draws/schedule=10000
deployment   : 100% SMR (large rides the loser channel: intl spillover + x * SMR program)


In [2]:
# --- Step 3 artifact guard: everything Step 3 shipped must be byte-identical after this
# notebook runs (the sensitivity columns REUSE those files; nothing may rewrite them).
# construction_schedules_mc.csv is deliberately in the list: the Export-2 cell below
# re-writes it from the same deterministic code, so equality doubles as a determinism check.
import hashlib

def _sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

GUARDED_STEP3_FILES = [
    REPO_ROOT / "cases_nuclearlearning_smr100.csv",
    REPO_ROOT / "inputs" / "financials" / "construction_schedules_mc.csv",
    REPO_ROOT / "inputs" / "financials" / "financials_tech_mc_smr100_eia_p50.csv",
    REPO_ROOT / "inputs" / "financials" / "construction_times_mc_large100_eo_p50.csv",
    REPO_ROOT / "inputs" / "plant_characteristics" / "nuclear_mc_smr100_eia_p50.csv",
    REPO_ROOT / "inputs" / "plant_characteristics" / "nuclear-smr_mc_large100_eo_p50.csv",
]
GUARD_SHA = {p: _sha(p) for p in GUARDED_STEP3_FILES}
print(f"snapshotted {len(GUARD_SHA)} Step 3 artifacts (verified unchanged in QA-S6)")


snapshotted 6 Step 3 artifacts (verified unchanged in QA-S6)


## Ported foundations

Verbatim from `smr100_case_export.ipynb` (itself porting `mc_cost_trajectories.ipynb`
S2–S7): schedules, PRIS experience stocks, the OCC engine, the INL duration model, the
ReEDS financing replication, the copula draw, and the program-NPV ranking.


In [3]:
US_GW_2024 = 97.0
OFFSET_GW = 3.0

US_SCHEDULES_CSV = REPO_ROOT / "inputs" / "nuclear_learning" / "US_SCHEDULES.csv"
_sched_df = pd.read_csv(US_SCHEDULES_CSV, comment="#", index_col="year")
TOKEN_NAME = {"eia_aeo_high": "EIA 2026 AEO high", "abou_jaoude": "Abou-Jaoude mod",
              "iaea_high": "IAEA high", "mckinsey": "McKinsey GEP 2025",
              "cop28": "COP28 tripling pledge", "eo2025": "2025 EO"}
assert set(_sched_df.columns) == set(TOKEN_NAME), "US_SCHEDULES.csv columns changed"
assert list(_sched_df.index) == list(YEARS), "US_SCHEDULES.csv year grid changed"
US_SCHEDULES = {TOKEN_NAME[tok]: _sched_df[tok].to_numpy(float) for tok in TOKEN_NAME}
SCHED_ORDER = list(US_SCHEDULES)
SCEN_TOKEN = {v: k for k, v in TOKEN_NAME.items()}

REGION_MILESTONES = {
    "CA":  (12.7, 12.7, 12.7, 18.8, 23.0, 30.0, 63.7),
    "SAM": (5.1, 5, 5, 8, 11, 8, 20),
    "WEU": (95.4, 86, 92, 81, 114, 68, 136),
    "EEU": (53.6, 57, 57, 58, 82, 68, 103),
    "AF":  (1.9, 4, 5, 7, 12, 12, 30),
    "WA":  (5.8, 11, 11, 12, 20, 15, 35),
    "SA":  (11.1, 18, 22, 25, 43, 45, 85),
    "CEA": (94.5, 136, 144, 215, 264, 242, 328),
    "SEA": (0, 0, 0, 3, 7, 4, 18),
    "OC":  (0, 0, 0, 0, 0, 0, 2),
}
THETA_KV = {"CA": 0.28, "SAM": 0.14, "WEU": 0.17794117647, "EEU": 0.1325, "AF": 0.16688725490,
            "WA": 0.13375, "SA": 0.185, "CEA": 0.135, "SEA": 0.16583333, "OC": 0.18}
REGIONS = list(THETA_KV.keys())

import pris_loader as pl

PRIS_UNITS = pl.load_units(str(NB_DIR / "rds2_2025_units.csv"))
_milestones_2024 = {r: REGION_MILESTONES[r][0] for r in REGIONS} | {"US": US_GW_2024}
_issues = pl.validate_units(PRIS_UNITS[PRIS_UNITS["status"] == "operational"], _milestones_2024)
assert not _issues, _issues
FLEET = PRIS_UNITS[PRIS_UNITS["status"] == "operational"]
UC_UNITS = PRIS_UNITS[PRIS_UNITS["status"] == "under construction"]

def retirement_flow_gw(key, life_yr):
    """GW retiring per model year, from each unit's real grid-connection date + a lifetime rule."""
    return pl.retirement_schedule(FLEET, pd.Index(YEARS), lifetime_years=life_yr,
                                  region=key).to_numpy()

PIPE_END = 2030   # pipeline's real visibility horizon; structurally zero after 2030 (mc S3)
PIPELINE_GW = {r: pl.committed_pipeline(UC_UNITS[UC_UNITS["region"] == r],
                                        pd.Index(np.arange(2025, PIPE_END + 1)),
                                        lead_time_months=72).to_dict()
               for r in REGIONS}
# Under-construction units are excluded: 5 carry PLANNED grid dates (not completed experience)
_ever = PRIS_UNITS[(PRIS_UNITS["status"] != "under construction")
                   & PRIS_UNITS["grid_connection"].notna()]
HIST_UNITS = {r: int((_ever["region"] == r).sum()) for r in REGIONS + ["US"]}
US_LICENSE_LIFE = 80.0
ONE_FACTOR_DELTA = 0.0

def net_path_gw(region, u):
    """Regional net capacity path: interpolate the 2024/2030/2040/2050 milestones, positioned
    between Low and High by the global deployment draw u."""
    c = REGION_MILESTONES[region]
    vals = [c[0], c[1] + u*(c[2]-c[1]), c[3] + u*(c[4]-c[3]), c[5] + u*(c[6]-c[5])]
    return np.interp(YEARS, [2024, 2030, 2040, 2050], vals)

def gross_additions_gw(region, u):
    """The retirement identity: builds = max(0, capacity change + retirements); the committed
    pipeline REPLACES the interpolated path through PIPE_END = 2030 (real units; structurally
    zero after 2030, so it only shapes the pre-anchor record - see mc_cost_trajectories S3)."""
    dnet = np.diff(net_path_gw(region, u), prepend=net_path_gw(region, u)[0])
    ret = retirement_flow_gw(region, 65.0 + 5.0*u)   # IAEA-calibrated lives, 65+5u (mc S3; 2026-08-05)
    g = np.clip(dnet + ret, 0.0, None)
    pipe_mask = YEARS <= PIPE_END
    g[pipe_mask] = [PIPELINE_GW[region].get(int(y), 0.0) for y in YEARS[pipe_mask]]
    return g

# Foreign stocks on the u grid (companion S3): unsplit, 1-GW basis - the engine's cross-firm
# channel does not distinguish designs (companion S5 Step 5). The IAEA SMR-share split of the
# foreign fleet lives in the main notebook's S14 own-routing sensitivity, not in any engine.
REGION_STOCK = {}
for r in REGIONS:
    m_a = np.empty((len(U_GRID), T))
    for gi, u in enumerate(U_GRID):
        flow = gross_additions_gw(r, u)
        flow = flow.copy(); flow[YEARS <= ANCHOR] = 0.0
        m_a[gi] = np.cumsum(flow) / UNIT_FOREIGN_GW
    REGION_STOCK[r] = m_a

# US gross GW added per year under each schedule (post-anchor), and the program unit
# counts under 100%-SMR (N_US_SMR) and 100%-large (N_US_LARGE, the large100 comparators)
# builds. ZEROS_T is the loser tech's (international-only) channel in either direction.
US_RET = retirement_flow_gw("US", US_LICENSE_LIFE)
GW_ADD, N_US_SMR, N_US_LARGE = {}, {}, {}
for name in SCHED_ORDER:
    cap = US_SCHEDULES[name] - OFFSET_GW
    dnet = np.diff(cap, prepend=cap[0])
    flow = np.clip(dnet + US_RET, 0.0, None)
    flow[YEARS <= ANCHOR] = 0.0
    GW_ADD[name] = flow
    N_US_SMR[name] = np.cumsum(flow) / TECH["smr"]["unit_gw"]
    N_US_LARGE[name] = np.cumsum(flow) / TECH["large"]["unit_gw"]
ZEROS_T = np.zeros(T)

print(f"PRIS fleet loaded: {len(FLEET)} operational, {len(UC_UNITS)} under construction")
print("SMR units by 2050 under 100%-SMR deployment, per schedule:")
for name in SCHED_ORDER:
    print(f"  {name:22s} {N_US_SMR[name][-1]:6.0f} SMR units ({GW_ADD[name].sum():.0f} GW)")

PRIS fleet loaded: 417 operational, 62 under construction
SMR units by 2050 under 100%-SMR deployment, per schedule:
  EIA 2026 AEO high          69 SMR units (21 GW)
  Abou-Jaoude mod           121 SMR units (36 GW)
  IAEA high                 252 SMR units (76 GW)
  McKinsey GEP 2025         345 SMR units (103 GW)
  COP28 tripling pledge     678 SMR units (203 GW)
  2025 EO                  1011 SMR units (303 GW)


In [4]:
# The OCC engine — verbatim companion port (mc_cost_trajectories.ipynb S5).
H_ALL_W = sum(THETA_KV[r]*HIST_UNITS[r] for r in REGIONS)

def lag1(a):
    """One-year completion lag along the time axis: the stock ENTERING year t is the builds
    completed through t-1. Cumulative arrays stay end-of-year for accounting; the lag is
    applied here, at the pricing boundary - without it the first post-anchor cohort would
    price on its own not-yet-built units (below BOAK). Companion-identical (mc S5)."""
    a = np.asarray(a, float)
    out = np.zeros_like(a)
    out[..., 1:] = a[..., :-1]
    return out

OTHER_TECH = {"large": "smr", "smr": "large"}
X_IN_COL = {"large": "x_sl", "smr": "x_ls"}     # incoming cross-tech fraction, per receiving tech

def experience_channels(world, tech, n_us, n_oth=None):
    """The two experience stocks O_d(t) (own) and A_d(t) (cross-firm), each (n_draws, T).
    Companion-identical (mc S5): international experience enters the CROSS-FIRM channel at
    weight s*theta, unsplit across technologies (the K&V flows are normalized to the
    domestic inter-firm citation baseline - omega prices the firm wall, theta the border);
    the drawn incoming x scales the other tech's US program (n_oth) into the same channel."""
    n = len(world)
    u_r = np.repeat(world["u"].values[:, None], len(REGIONS), axis=1)
    if ONE_FACTOR_DELTA > 0:
        u_r = np.clip(u_r + rng_stream("one_factor_noise").uniform(
            -ONE_FACTOR_DELTA, ONE_FACTOR_DELTA, (n, len(REGIONS))), 0, 1)
    gi = np.clip(np.rint(u_r * (len(U_GRID)-1)).astype(int), 0, len(U_GRID)-1)
    S_kv = np.zeros((n, T))
    for j, r in enumerate(REGIONS):
        S_kv += THETA_KV[r] * REGION_STOCK[r][gi[:, j], :]
    S_kv = lag1(S_kv)                                    # stock entering year t (built thru t-1)
    N_us = lag1(np.broadcast_to(np.asarray(n_us, float), (n, T)))
    conv = world["conv_full"].values[:, None].astype(float)
    m = world["n_vendors"].values[:, None].astype(float)
    s = world["s"].values[:, None]
    hist_us = HIST_UNITS["US"] if tech == "large" else 0.0    # D8': US LWR history -> large only
    own0 = N_BOAK_UNITS + conv*hist_us/m
    own = own0 + N_us/m
    oth = (m-1.0)*own0 + N_us*(m-1.0)/m \
          + s*(conv*H_ALL_W + S_kv)
    if n_oth is not None:                                # cross-tech: x * the other tech's program
        x = world[X_IN_COL[tech]].values[:, None]
        oth = oth + x*lag1(np.broadcast_to(np.asarray(n_oth, float), (n, T)))
    return own, oth

CES_EPS = 1e-8

def occ_paths_ces(world, tech, n_us, n_oth=None, ces_rho=None):
    """Production OCC engine (companion D11/D12, verbatim). Returns $/kW (2022 USD)."""
    own, oth = experience_channels(world, tech, n_us, n_oth=n_oth)
    assert own.min() >= N_BOAK_UNITS and oth.min() >= N_BOAK_UNITS, "stock below anchor base; log unsafe"
    lr = world[f"lr_{tech}"].values[:, None]
    b1, b2 = np.log2(1.0 - lr), np.log2(1.0 - OMEGA*lr)
    b, w = -(b1 + b2), b1/(b1 + b2)
    rho = world["ces_rho"].values if ces_rho is None else np.full(len(world), float(ces_rho))
    rho = rho[:, None]
    geo = np.abs(rho) < CES_EPS
    lnO, lnA = np.log(own), np.log(oth)
    with np.errstate(over="raise"):
        lnE = np.where(geo, w*lnO + (1.0-w)*lnA,
                       lnO + np.log1p((1.0-w)*np.expm1(np.where(geo, 0.0, rho)*(lnA - lnO)))
                           / np.where(geo, 1.0, rho))
    return world[f"boak_{tech}"].values[:, None] * np.exp(-b*(lnE - lnE[:, [yi(ANCHOR)]]))

def percentile_table(paths, ps=(5, 25, 50, 75, 95)):
    """Yearly percentiles of a (n_draws, T) trajectory array."""
    return pd.DataFrame({f"P{p}": np.percentile(paths, p, axis=0) for p in ps}, index=YEARS)

In [5]:
# Durations + ReEDS financing — verbatim companion ports (S6-S7 there).
DUR_UNITS = pl.duration_panel(PRIS_UNITS)
DUR_UNITS = DUR_UNITS[DUR_UNITS["family"].isin(["AP1000", "APR1400", "HPR1000"])].reset_index(drop=True)
DUR_UNITS = DUR_UNITS.rename(columns={"reactor_name": "unit"})

def fit_family_fe(df):
    """ln(duration) = family intercepts + b*ln(family sequence); returns (coefficients, residual sd)."""
    X = np.column_stack([np.ones(len(df)),
                         (df["family"] == "APR1400").astype(float),
                         (df["family"] == "HPR1000").astype(float),
                         np.log(df["family_seq"].astype(float))])
    Y = np.log(df["months"].values)
    beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
    resid = Y - X @ beta
    return beta, resid.std(ddof=min(4, len(df)-1))

beta_hat, sig_hat = fit_family_fe(DUR_UNITS)

INL_DUR_MOD = np.array([118.0, 88.0, 74.0, 67.0, 62.0, 59.0, 56.0, 54.0, 52.0, 50.0])
INL_DUR_OPT = np.array([118.0, 70.0, 57.0, 49.0, 45.0, 42.0, 40.0, 38.0, 37.0, 36.0])
SMR_DUR_RATIO = 55.0/82.0
SMR_DUR_FLOOR = 43.0

def duration_paths(world, tech, n_us):
    """Construction duration (months, FNC -> COD) per draw-year, from the INL series curves."""
    n = len(world)
    N_us = lag1(np.broadcast_to(np.asarray(n_us, float), (n, T)))   # completed units only
    n_own = N_us / world["n_vendors"].values[:, None]
    series = np.clip(2 + np.floor(n_own/2.0).astype(int), 2, len(INL_DUR_MOD))
    opt, mod = np.take(INL_DUR_OPT, series-1), np.take(INL_DUR_MOD, series-1)
    base = opt + world["dur_lambda"].values[:, None]*(mod - opt)
    if tech == "smr":
        base = np.maximum(base * SMR_DUR_RATIO, SMR_DUR_FLOOR)
    return base * np.exp(world["dur_z"].values[:, None]*sig_hat)

FIN_DIR = REPO_ROOT / "inputs" / "financials"

def load_reeds_financials():
    """ReEDS financial inputs on the YEARS grid, exactly as reeds/financials.py derives them."""
    sys_fin = pd.read_csv(FIN_DIR / "financials_sys_ATB2024.csv")
    infl = pd.read_csv(FIN_DIR / "inflation_default.csv")
    sys_fin = sys_fin.merge(infl, on="t", how="left")
    sys_fin["d_nom"] = ((1 - sys_fin["debt_fraction"]) * (sys_fin["rroe_nom"] - 1)
                        + sys_fin["debt_fraction"] * (sys_fin["interest_rate_nom"] - 1)
                          * (1 - sys_fin["tax_rate"]) + 1)
    sys_fin["d_real"] = sys_fin["d_nom"] / sys_fin["inflation_rate"]
    tech_fin = pd.read_csv(FIN_DIR / "financials_tech_ATB2024.csv")
    tech_fin = tech_fin[tech_fin["i"].isin(["Nuclear", "Nuclear-SMR"])]
    dep = pd.read_csv(FIN_DIR / "depreciation_schedules_default.csv")
    cs = pd.read_csv(FIN_DIR / "construction_schedules_default.csv")

    def on_years(col):
        s = sys_fin.set_index("t")[col].reindex(range(1990, YEARS[-1] + 1)).ffill()
        return s.loc[YEARS].to_numpy(float)

    out = {"interest_base": on_years("interest_rate_nom"), "tax_rate": on_years("tax_rate"),
           "d_nom": on_years("d_nom"), "d_real": on_years("d_real"),
           "pv_dep": {}, "risk_mult": {}, "eval_adj": {}, "sched": {}}
    SYS_EVAL_YEARS = 30
    sys_pvf_sum = (1 - (1/out["d_real"])**(SYS_EVAL_YEARS - 1)) / (out["d_real"] - 1.0) + 1
    REEDS_TECH_NAME = {"large": "Nuclear", "smr": "Nuclear-SMR"}
    for tech, iname in REEDS_TECH_NAME.items():
        row = tech_fin[tech_fin["i"] == iname].iloc[-1]
        dep_frac = dep[str(int(row["depreciation_sch"]))].to_numpy(float)
        out["pv_dep"][tech] = np.array([np.sum(dep_frac / dn**np.arange(1, 22))
                                        for dn in out["d_nom"]])
        eval_p = float(row["eval_period"])
        out["risk_mult"][tech] = 1.0 + float(row["finance_diff_real"]) * (
            (1 - (1/out["d_real"])**eval_p) / (out["d_real"] - 1.0))
        tech_pvf_sum = (1 - (1/out["d_real"])**(eval_p - 1)) / (out["d_real"] - 1.0) + 1
        out["eval_adj"][tech] = sys_pvf_sum / tech_pvf_sum
        frac = pd.to_numeric(cs[str(row["construction_sch"])], errors="coerce").fillna(0.0).to_numpy()
        nz = np.flatnonzero(frac > 0)
        assert nz.size and nz[-1] - nz[0] + 1 == nz.size, \
            f"{iname}: spend profile has interior zeros; frac[frac>0] would shift spend years"
        out["sched"][tech] = frac[nz[0]:nz[-1] + 1]
    return out

FIN = load_reeds_financials()

def _resample_schedule(frac, n_years):
    """Stretch/compress a spend-fraction profile to `n_years` bins (sum to 1)."""
    frac = np.asarray(frac, float)
    n0 = len(frac)
    if n_years == n0:
        return frac / frac.sum()
    cdf = np.concatenate([[0.0], np.cumsum(frac)])
    xq = np.linspace(0.0, 1.0, n_years + 1)
    cdf_q = np.interp(xq, np.linspace(0.0, 1.0, n0 + 1), cdf)
    new = np.diff(cdf_q)
    return new / new.sum()

def ccmult_from_duration(duration_mo, interest_base, canonical_frac):
    """Construction-financing (IDC) multiplier for a learned duration (verbatim ReEDS port)."""
    n_years = int(round(duration_mo / 12.0))
    n_years = max(1, min(n_years, 10))
    x = _resample_schedule(canonical_frac, n_years)
    exps = np.arange(n_years) + 0.5
    return 1.0 + float(np.sum(x * (interest_base ** exps - 1.0)))

def ccmult_grid(dur_months, tech):
    """Vectorized ccmult for a (n_draws, T) duration array."""
    ib = FIN["interest_base"]
    table = np.empty((10 + 1, T))
    for n in range(1, 11):
        x = _resample_schedule(FIN["sched"][tech], n)
        exps = np.arange(n) + 0.5
        table[n] = 1.0 + (x[:, None] * (ib[None, :] ** exps[:, None] - 1.0)).sum(axis=0)
    n_years = np.clip(np.round(np.asarray(dur_months)/12.0).astype(int), 1, 10)
    return table[n_years, np.arange(T)[None, :]]

def fin_mult_rest(tech, use_itc=False):
    """Everything in ReEDS's cost_cap_fin_mult EXCEPT ccmult, per year (T,) — verbatim port."""
    assert not use_itc, "ITC variant deliberately not implemented"
    tax, pv_dep = FIN["tax_rate"], FIN["pv_dep"][tech]
    return (1.0/(1.0 - tax)) * (1.0 - tax*pv_dep) * FIN["risk_mult"][tech] * FIN["eval_adj"][tech]

FIN_REST = {tech: fin_mult_rest(tech) for tech in TECH}
DISC = np.cumprod(np.where(YEARS > ANCHOR, FIN["d_real"], 1.0))   # discount to 2030
print(f"foundations loaded: duration panel {len(DUR_UNITS)} units (sd {sig_hat:.3f}); "
      f"ccmult(72mo, 8%, '6') = {ccmult_from_duration(72.0, 1.08, FIN['sched']['large']):.6f}")

foundations loaded: duration panel 21 units (sd 0.116); ccmult(72mo, 8%, '6') = 1.268124

In [6]:
# Ported verbatim from mc_cost_trajectories.ipynb S8.
RHO_SETS = {"zero": 0.0, "moderate": -0.3, "strong": -0.6}
LATENTS = ["lr", "boak", "u"]

def corr_matrix(rho_key):
    C = np.eye(3)
    C[0, 1] = C[1, 0] = RHO_SETS[rho_key]
    ev = np.linalg.eigvalsh(C)
    assert ev.min() > 1e-10, f"correlation matrix '{rho_key}' is not positive definite: {ev}"
    return C

def draw_world(n, rho_key, rng):
    """Draw n worlds: every uncertain input, one row per draw (companion S8, verbatim)."""
    Z = rng.standard_normal((n, 3)) @ np.linalg.cholesky(corr_matrix(rho_key)).T
    U = stats.norm.cdf(Z)                                   # Gaussian copula -> uniforms
    w = pd.DataFrame(index=range(n))
    for tech in TECH:
        t = TECH[tech]
        w[f"lr_{tech}"] = t["lr_lo"] + U[:, 0]*(t["lr_hi"] - t["lr_lo"])      # comonotone across techs
        w[f"boak_{tech}"] = t["boak_lo"] + U[:, 1]*(t["boak_hi"] - t["boak_lo"])
    w["u"] = U[:, 2]                                        # global deployment position
    w["s"] = rng.uniform(0.0, 1.0, n)                       # spillover scale
    w["n_vendors"] = rng.integers(4, 9, n)                  # vendor count m
    w["conv_full"] = rng.integers(0, 2, n)                  # 0 = tiny-base, 1 = full-stock
    w["ces_rho"] = rng.choice(CES_RHO_GRID, size=n)         # channel-substitution stratum
    w["dur_lambda"] = rng.uniform(0.0, 1.0, n)              # duration scenario position
    w["dur_z"] = rng.standard_normal(n)                     # persistent duration project noise
    w["x_ls"] = rng.uniform(0.0, X_SPILL_MAX, n)            # cross-tech spillover: large -> SMR
    w["x_sl"] = rng.uniform(0.0, X_SPILL_MAX, n)            # cross-tech spillover: SMR -> large
    return w

WORLDS, results = {}, {}
for sched in SCHED_ORDER:
    # The COMPANION's stream names, deliberately: identical worlds to mc_cost_trajectories.ipynb
    world = draw_world(N_DRAWS, COPULA_SET, rng_stream(f"world/{SCEN_TOKEN[sched]}"))
    WORLDS[sched] = world
    r = {}
    for tech, n_us, n_oth in (("smr", N_US_SMR[sched], None),
                              ("large", ZEROS_T, N_US_SMR[sched])):   # loser: x * SMR program
        r[f"occ_{tech}"] = occ_paths_ces(world, tech, n_us, n_oth=n_oth)
        r[f"dur_{tech}"] = duration_paths(world, tech, n_us)
        r[f"ccmult_{tech}"] = ccmult_grid(r[f"dur_{tech}"], tech)
        r[f"fincapex_{tech}"] = r[f"occ_{tech}"] * r[f"ccmult_{tech}"] * FIN_REST[tech][None, :]
    results[sched] = r

print(f"drew {N_DRAWS} worlds x {len(SCHED_ORDER)} schedules (companion draw, comonotone)")
print("\n2050 SMR OCC and financed CAPEX by schedule ($/kW 2022, P5 / P50 / P95):")
for sched in SCHED_ORDER:
    o = results[sched]["occ_smr"][:, yi(2050)]
    f = results[sched]["fincapex_smr"][:, yi(2050)]
    print(f"  {sched:22s} OCC {np.percentile(o,5):5,.0f} /{np.percentile(o,50):6,.0f} /"
          f"{np.percentile(o,95):6,.0f}   financed {np.percentile(f,5):5,.0f} /"
          f"{np.percentile(f,50):6,.0f} /{np.percentile(f,95):6,.0f}")

drew 10000 worlds x 6 schedules (companion draw, comonotone)

2050 SMR OCC and financed CAPEX by schedule ($/kW 2022, P5 / P50 / P95):
  EIA 2026 AEO high      OCC 3,480 / 5,494 / 8,151   financed 4,597 / 7,280 /10,831
  Abou-Jaoude mod        OCC 3,101 / 5,110 / 7,897   financed 4,114 / 6,779 /10,432
  IAEA high              OCC 2,506 / 4,493 / 7,441   financed 3,307 / 5,967 / 9,878
  McKinsey GEP 2025      OCC 2,304 / 4,291 / 7,277   financed 3,067 / 5,705 / 9,681
  COP28 tripling pledge  OCC 1,891 / 3,779 / 6,890   financed 2,505 / 5,016 / 9,134
  2025 EO                OCC 1,693 / 3,483 / 6,610   financed 2,251 / 4,615 / 8,776


In [7]:
# --- Non-CAPEX costs for the NPV ranking (adopted 2026-08-03) ---
# All validation lives in npv_winner_check.ipynb: the winner survives the full NPV in >99%
# of draws, but WHICH draw sits nearest each P5/P50/P95 target moves -- so the selection
# here carries the full NPV. Convention: FOM and VOM sit at the same percentile of their
# ranges (ATB 2024 advanced -> conservative) as the draw's 2030 anchor cost does in its
# range -- comonotone with the cost dial, never independent sensitivities. CF equals ReEDS's
# own availability (avail, replicated bit-exactly from raw inputs in npv_winner_check N1);
# fuel is heat rate x AEO 2026 uranium; annual costs are levelized with ReEDS's
# pvf_onm = 1/crf (30-yr window at the system real rate).
PLANTCHAR_DIR = REPO_ROOT / "inputs" / "plant_characteristics"
EVAL_YEARS = 30
_d_real = float(DISC[-1] / DISC[-2])
_ks = np.arange(1, EVAL_YEARS + 1)
PVF_ONM = float((_d_real ** -_ks).sum())
assert round(1.0 / PVF_ONM, 5) == 0.06773   # ReEDS crf.csv pin (validated in npv_winner_check)

# CF: the avail number computed and QA'd by npv_winner_check.ipynb (national mean; identical
# for both techs by ReEDS's prime-mover construction)
_cf = pd.read_csv(NB_DIR / "exports" / "npv_winner_summary.csv")["CF"].unique()
assert len(_cf) == 1 and 0.89 < float(_cf[0]) < 0.92
CF_AVAIL = float(_cf[0])

_ATB_STEM = {"large": "nuclear", "smr": "nuclear-smr"}
OM_RANGE = {}
for tech in TECH:
    ends = {s: pd.read_csv(PLANTCHAR_DIR / f"{_ATB_STEM[tech]}_ATB_2024_{s}.csv")
                 .set_index("t").loc[2030]
            for s in ("advanced", "conservative")}
    # the ATB scenario axis IS the MC's anchor-cost axis -- one percentile indexes both
    assert (float(ends["advanced"]["capcost"]), float(ends["conservative"]["capcost"])) \
        == (TECH[tech]["boak_lo"], TECH[tech]["boak_hi"]), tech
    assert float(ends["advanced"]["heatrate"]) == float(ends["conservative"]["heatrate"])
    OM_RANGE[tech] = {
        "fom": (float(ends["advanced"]["fom"]), float(ends["conservative"]["fom"])),
        "vom": (float(ends["advanced"]["vom"]), float(ends["conservative"]["vom"])),
        "hr": float(ends["advanced"]["heatrate"]),
    }

def u1_of(world):
    """The draw's cost-dial percentile (shared by both techs -- comonotone)."""
    t = TECH["large"]
    return ((world["boak_large"] - t["boak_lo"]) / (t["boak_hi"] - t["boak_lo"])).to_numpy()

def om_at(tech, U1):
    """FOM ($/kW-yr) and VOM ($/MWh) at cost-dial percentile U1 (advanced -> conservative)."""
    lo_f, hi_f = OM_RANGE[tech]["fom"]
    lo_v, hi_v = OM_RANGE[tech]["vom"]
    return lo_f + U1 * (hi_f - lo_f), lo_v + U1 * (hi_v - lo_v)

# uranium AEO 2026 baseline: 2025$ -> 2022$, flat after 2050; per-build-year 30-yr fuel PV
_defl = pd.read_csv(REPO_ROOT / "inputs" / "financials" / "deflator.csv")
_defl.columns = ["year", "deflator"]
_defl = _defl.set_index("year")["deflator"]
_Pu = (pd.read_csv(REPO_ROOT / "inputs" / "fuelprices" / "uranium_AEO_2026_baseline.csv")
       .set_index("year")["cost"] * float(_defl.loc[2025] / _defl.loc[2022]))
_Pu = _Pu.reindex(range(int(YEARS.min()), int(YEARS.max()) + EVAL_YEARS + 1)).ffill()
MWH_PER_KWYR = 8.760 * CF_AVAIL                    # MWh generated per kW-yr at CF = avail
_FUEL_PV = {tech: MWH_PER_KWYR * OM_RANGE[tech]["hr"] * np.array(
    [(_Pu.loc[t + 1: t + EVAL_YEARS].to_numpy() * (_d_real ** -_ks)).sum() for t in YEARS])
    for tech in TECH}

def om_npv_per_kw(tech, U1):
    """(n, T) 30-yr PV per kW of capacity built in each year: FOM + VOM + fuel, 2022$."""
    fom, vom = om_at(tech, np.asarray(U1)[:, None])
    return PVF_ONM * (fom + MWH_PER_KWYR * vom) + _FUEL_PV[tech][None, :]

# the convention's check values (SMR at U1 = 0.75): boak $8,875/kW -> FOM $191.5, VOM $2.65
_f, _v = om_at("smr", 0.75)
assert (round(float(_f), 1), round(float(_v), 2)) == (191.5, 2.65)
print(f"NPV-ranking inputs: CF = avail = {CF_AVAIL:.4f} (npv_winner_check), "
      f"pvf_onm = 1/crf = {PVF_ONM:.4f};")
print("  FOM/VOM comonotone with the draw's cost percentile across ATB advanced->conservative;")
print("  30-yr PV of non-CAPEX per kW built 2031, at U1=0.5 ($/kW 2022): "
      f"large {om_npv_per_kw('large', np.array([0.5]))[0, yi(2031)]:,.0f}, "
      f"smr {om_npv_per_kw('smr', np.array([0.5]))[0, yi(2031)]:,.0f}")


NPV-ranking inputs: CF = avail = 0.9044 (npv_winner_check), pvf_onm = 1/crf = 14.7637;
  FOM/VOM comonotone with the draw's cost percentile across ATB advanced->conservative;
  30-yr PV of non-CAPEX per kW built 2031, at U1=0.5 ($/kW 2022): large 3,303, smr 3,245


In [8]:
SHORT = {"eia_aeo_high": "eia", "abou_jaoude": "aj", "iaea_high": "iaea",
         "mckinsey": "mck", "cop28": "cop28", "eo2025": "eo"}

def rank_smr(sched):
    """Program-NPV score per draw: discounted GW-weighted SMR financed CAPEX plus the 30-yr
    PV of FOM/VOM/fuel at the draw's own O&M percentile (lower = cheaper)."""
    w_t = GW_ADD[sched] / DISC
    npv = results[sched]["fincapex_smr"] + om_npv_per_kw("smr", u1_of(WORLDS[sched]))
    return npv @ w_t

print("cost object defined: rank_smr(sched) — the program-NPV scale every diagnostic, "
      "bound, and placement below is measured on")

cost object defined: rank_smr(sched) — the program-NPV scale every diagnostic, bound, and placement below is measured on


In [9]:
# The per-schedule program-NPV score vectors. In smr100_case_export.ipynb this line lives
# inside the OAT/monotonicity cell; the bounds/monotonicity machinery itself is NOT ported
# (it is documentation of the Step-1 case selection, already frozen -- see that notebook).
mc_score = {s: rank_smr(s) for s in SCHED_ORDER}
print("program-NPV scores computed for all six schedules")


program-NPV scores computed for all six schedules


## Base-case selection, re-derived and pinned

The smr100 P5/P50/P95 selection is re-derived by the ported cell (writing this notebook's
own copy under `exports/step4/`) and then asserted identical to the frozen Step-1
registration in `exports/smr100/selected_draws.csv` — the sensitivity columns point at
those cases' files, so the identity pin is what licenses the reuse.


In [10]:
# --- Production-case selection (v10, 2026-08-06): the P5/P50/P95 joint draws per schedule ---
# Rank the 10k drawn worlds by the program NPV (mc_score) and take the draws at the
# P5/P50/P95 ranks (rank = ceil(q*(N-1)) of the stable argsort). The cases are actual joint
# draws -- plausible worlds with probability mass -- selected as a constrained optimizer over
# the drawn (plausible) set; the lo/hi bounds above remain the appendix possibility frontier
# over the priors' full support.
PCT_TAGS = {"p05": 0.05, "p50": 0.50, "p95": 0.95}
SELECTED = {}                # SELECTED[(sched, tag)] -> {"idx": draw index, "score": float}
_sel_rows = []
for sched in SCHED_ORDER:
    order = np.argsort(mc_score[sched], kind="stable")
    for tag, q in PCT_TAGS.items():
        idx = int(order[int(np.ceil(q * (N_DRAWS - 1)))])
        SELECTED[(sched, tag)] = {"idx": idx, "score": float(mc_score[sched][idx])}
        _sel_rows.append({"schedule": sched, "percentile": tag, "draw_index": idx,
                          "score": float(mc_score[sched][idx]),
                          "pctile_in_MC": round(100*float(
                              (mc_score[sched] < mc_score[sched][idx]).mean()), 3),
                          **{c: float(WORLDS[sched].iloc[idx][c])
                             for c in WORLDS[sched].columns}})
sel_df = pd.DataFrame(_sel_rows).set_index(["schedule", "percentile"])
sel_df.to_csv(EXPORTS / "selected_draws.csv")

# Draw-identity assert: each selected draw's parameter row must match the companion's
# per-draw export (extends QA-0's sample-space identity to the cases themselves).
_npz_sel_path = NB_DIR / "exports" / "mc_perdraw.npz"
_sel_checked = 0
if _npz_sel_path.exists():
    _npz_sel = np.load(_npz_sel_path, allow_pickle=False)
    _sel_cols = list(_npz_sel["world_columns"].astype(str))
    for (sched, tag), s in SELECTED.items():
        tok = SCEN_TOKEN[sched]
        if _npz_sel[f"worlds_{tok}"].shape[0] != N_DRAWS:
            print("selected-draw identity SKIPPED: mc_perdraw.npz draw count differs "
                  "from this run -- re-run the companion notebook to compare")
            break
        assert _sel_cols == list(WORLDS[sched].columns), "world column layout diverged"
        assert np.allclose(_npz_sel[f"worlds_{tok}"][s["idx"]],
                           WORLDS[sched].iloc[s["idx"]].to_numpy(dtype=float),
                           rtol=1e-9, atol=0), (sched, tag)
        _sel_checked += 1
    if _sel_checked:
        print(f"selected-draw identity vs mc_perdraw.npz: all {_sel_checked} case rows match "
              "their registered draw (rtol 1e-9)")
else:
    print("mc_perdraw.npz not found -- selected-draw identity check skipped "
          "(run the companion notebook to enable it)")

print("\nSelected production draws (registered to exports/smr100/selected_draws.csv):")
print(sel_df[["draw_index", "score", "pctile_in_MC"]].to_string())

selected-draw identity vs mc_perdraw.npz: all 18 case rows match their registered draw (rtol 1e-9)

Selected production draws (registered to exports/smr100/selected_draws.csv):
                                  draw_index         score  pctile_in_MC
schedule              percentile                                        
EIA 2026 AEO high     p05               2329  1.074627e+05           5.0
                      p50               9573  1.481451e+05          50.0
                      p95               6874  1.950652e+05          95.0
Abou-Jaoude mod       p05               6077  1.399636e+05           5.0
                      p50                242  1.971983e+05          50.0
                      p95                162  2.659700e+05          95.0
IAEA high             p05               8423  3.268254e+05           5.0
                      p50               4993  4.708006e+05          50.0
                      p95               7602  6.512874e+05          95.0
McKinsey GEP 2025   

In [11]:
# --- Continuity pin: the re-derived smr100 selection == the frozen Step-1 registration ---
_frozen_sel = pd.read_csv(NB_DIR / "exports" / "smr100" / "selected_draws.csv",
                          index_col=["schedule", "percentile"])
assert list(_frozen_sel.index) == list(sel_df.index)
assert (_frozen_sel["draw_index"] == sel_df["draw_index"]).all(), "draw indices diverged"
_fcols = [c for c in _frozen_sel.columns if c != "draw_index"]
assert np.allclose(_frozen_sel[_fcols].to_numpy(float), sel_df[_fcols].to_numpy(float),
                   rtol=1e-9, atol=0), "selection values diverged from the frozen registration"
print("base-selection pin PASSED: all 18 smr100 selections identical to "
      "exports/smr100/selected_draws.csv (draw indices exact, values rtol 1e-9)")


base-selection pin PASSED: all 18 smr100 selections identical to exports/smr100/selected_draws.csv (draw indices exact, values rtol 1e-9)


In [12]:
# --- The 18 exported cases, built from the selected P5/P50/P95 joint draws ---
N_US_PROG = {"smr": N_US_SMR, "large": N_US_LARGE}   # program-tech unit counts per schedule

def build_case_rec(case, sched, tag, w1, score, extra, program="smr", score_dist=None):
    """One export-ready case record: parameters, U1-comonotone O&M, and the (T,) occ/dur
    path arrays -- shared by the production cases, the pilot bounds pair (no ATB-moderate
    special case anymore, v10), and the large100 comparators. `program` is the tech that
    receives the entire build program (the 'winner' field records it; the other tech rides
    the loser channel); `score_dist` is the ranking the case's percentile is measured in
    (defaults to the SMR program ranking mc_score)."""
    fomvom = {t: tuple(float(x) for x in om_at(t, float(u1_of(w1)[0]))) for t in TECH}
    r = w1.iloc[0]
    dist = mc_score[sched] if score_dist is None else score_dist
    rec = {"case": case, "schedule": sched, "scen_token": SCEN_TOKEN[sched], "case_type": tag,
           "winner": program,          # the tech assumed to receive the entire program
           "lr_large": float(r["lr_large"]), "lr_smr": float(r["lr_smr"]),
           "boak_large": float(r["boak_large"]), "boak_smr": float(r["boak_smr"]),
           "u": float(r["u"]), "s_spill": float(r["s"]),
           "x_ls": float(r["x_ls"]), "x_sl": float(r["x_sl"]),
           "n_vendors": int(r["n_vendors"]),
           "convention": "full-stock" if r["conv_full"] else "tiny-base",
           "ces_rho": float(r["ces_rho"]), "dur_lambda": float(r["dur_lambda"]),
           "dur_z": float(r["dur_z"]),
           "score": float(score),
           "pctile_in_MC": round(100*float((dist < score).mean()), 2),
           **extra,
           **{f"{k}_{t}": v for t in TECH for k, v in zip(("fom", "vom"), fomvom[t])}}
    other = "large" if program == "smr" else "smr"
    for tech, n_us, n_oth in ((program, N_US_PROG[program][sched], None),
                              (other, ZEROS_T, N_US_PROG[program][sched])):
        occ = occ_paths_ces(w1, tech, n_us, n_oth=n_oth)[0]
        dur = duration_paths(w1, tech, n_us)[0]
        rec[f"occ_{tech}"], rec[f"dur_{tech}"] = occ, dur    # (T,) arrays for the exports
        for y in (2030, 2040, 2050):
            rec[f"occ_{tech}_{y}"] = round(float(occ[yi(y)]), 0)
            rec[f"dur_{tech}_{y}"] = round(float(dur[yi(y)]), 0)
    return rec

CASES = {}           # CASES[case_name] -> the selected joint draw's record (+ (T,) arrays)
for sched in SCHED_ORDER:
    tok = SCEN_TOKEN[sched]
    for tag in ("p05", "p50", "p95"):
        case = f"smr100_{SHORT[tok]}_{tag}"
        idx = SELECTED[(sched, tag)]["idx"]
        w1 = WORLDS[sched].iloc[[idx]].reset_index(drop=True)
        CASES[case] = build_case_rec(case, sched, tag, w1,
                                     SELECTED[(sched, tag)]["score"], {"draw_index": idx})

selected_cases = pd.DataFrame({c: {k: v for k, v in rec.items() if not isinstance(v, np.ndarray)}
                               for c, rec in CASES.items()}).T.set_index("case")
selected_cases.to_csv(EXPORTS / "selected_cases.csv")
print(f"built {len(CASES)} production cases (6 schedules x p05/p50/p95); "
      "full record frozen to exports/smr100/selected_cases.csv")
selected_cases[["schedule", "case_type", "draw_index", "pctile_in_MC", "lr_smr", "boak_smr",
                "n_vendors", "convention", "occ_smr_2050", "occ_large_2050"]]

built 18 production cases (6 schedules x p05/p50/p95); full record frozen to exports/smr100/selected_cases.csv


,schedule,case_type,draw_index,pctile_in_MC,lr_smr,boak_smr,n_vendors,convention,occ_smr_2050,occ_large_2050
case,,,,,,,,,,
smr100_eia_p05,EIA 2026 AEO high,p05,2329,5.0,0.158923,6417.436975,7,tiny-base,3119.0,3837.0
smr100_eia_p50,EIA 2026 AEO high,p50,9573,50.0,0.080338,7565.05389,4,full-stock,5609.0,6357.0
smr100_eia_p95,EIA 2026 AEO high,p95,6874,95.0,0.126522,9732.341093,4,full-stock,7634.0,7374.0
smr100_aj_p05,Abou-Jaoude mod,p05,6077,5.0,0.144374,6457.191383,4,tiny-base,2862.0,4332.0
smr100_aj_p50,Abou-Jaoude mod,p50,242,50.0,0.04183,7048.38307,6,full-stock,5870.0,6107.0
smr100_aj_p95,Abou-Jaoude mod,p95,162,95.0,0.053308,9689.143019,6,full-stock,7666.0,7570.0
smr100_iaea_p05,IAEA high,p05,8423,5.0,0.09802,5550.338622,4,tiny-base,2839.0,4968.0
smr100_iaea_p50,IAEA high,p50,4993,50.0,0.127165,9263.333245,4,tiny-base,3766.0,5047.0
smr100_iaea_p95,IAEA high,p95,7602,95.0,0.032521,9137.897016,7,tiny-base,7616.0,7250.0


## The large100 arm, completed to percentiles

The pure-large ranking (`rank_large` over the identical worlds) is ported verbatim,
including its P50 selection and the `tech_comparison` κ=1 pin. A new cell then extends the
selection to P5/P95 (same rank rule), pins the re-derived P50 draws to the Step-3 hard pin
and the frozen `selected_draws_large.csv`, and registers all three percentiles to
`exports/step4/selected_draws_large.csv`. Only the P5/P95 cases get input files — the P50
runs completed in Step 3.


In [13]:
# --- large100 ranking + P50 selection: the pure-large program over the identical worlds ---
# No new draws and no new RNG streams: rank_large re-scores the SAME worlds with the roles
# swapped, so the 18-case selection above is untouched by construction.
results_large, mc_score_large = {}, {}
for sched in SCHED_ORDER:
    occ = occ_paths_ces(WORLDS[sched], "large", N_US_LARGE[sched])
    dur = duration_paths(WORLDS[sched], "large", N_US_LARGE[sched])
    fincapex = occ * ccmult_grid(dur, "large") * FIN_REST["large"][None, :]
    results_large[sched] = {"occ_large": occ, "dur_large": dur, "fincapex_large": fincapex}
    mc_score_large[sched] = (fincapex + om_npv_per_kw("large", u1_of(WORLDS[sched]))) \
        @ (GW_ADD[sched] / DISC)

SELECTED_LARGE = {}
_sel_rows_large = []
for sched in SCHED_ORDER:
    order = np.argsort(mc_score_large[sched], kind="stable")
    idx = int(order[int(np.ceil(0.50 * (N_DRAWS - 1)))])
    SELECTED_LARGE[sched] = {"idx": idx, "score": float(mc_score_large[sched][idx])}
    _sel_rows_large.append({"schedule": sched, "percentile": "p50", "draw_index": idx,
                            "score": float(mc_score_large[sched][idx]),
                            "pctile_in_MC": round(100*float(
                                (mc_score_large[sched] < mc_score_large[sched][idx]).mean()), 3),
                            **{c: float(WORLDS[sched].iloc[idx][c])
                               for c in WORLDS[sched].columns}})
sel_large_df = pd.DataFrame(_sel_rows_large).set_index(["schedule", "percentile"])
sel_large_df.to_csv(EXPORTS / "selected_draws_large.csv")

# QA pin vs tech_comparison's kappa=1 per-draw export (the same ranking, computed there):
# world identity by column name, score identity, and P50-rank identity with a near-tie
# fallback for cross-kernel float noise (the in-notebook selection is authoritative).
_k100_dir = NB_DIR / "exports" / "tech_comparison" / "perdraw"
_k100_checked = 0
for sched in SCHED_ORDER:
    _p = _k100_dir / f"k100_{SCEN_TOKEN[sched]}.npz"
    if not _p.exists():
        print(f"k100 pin SKIPPED ({SCEN_TOKEN[sched]}): {_p.name} not found -- run "
              "tech_comparison.ipynb to enable it")
        continue
    _z = np.load(_p, allow_pickle=False)
    if _z["R_npv_large"].shape[0] != N_DRAWS:
        print(f"k100 pin SKIPPED ({SCEN_TOKEN[sched]}): draw count differs from this run")
        continue
    _zcols = list(_z["columns"].astype(str))
    _wcols = list(WORLDS[sched].columns)
    assert set(_wcols) <= set(_zcols), f"k100 world columns diverged ({SCEN_TOKEN[sched]})"
    assert np.allclose(_z["world"][:, [_zcols.index(c) for c in _wcols]],
                       WORLDS[sched].to_numpy(dtype=float), rtol=1e-9, atol=0), sched
    assert np.allclose(mc_score_large[sched], _z["R_npv_large"], rtol=1e-6), (
        f"rank_large diverged from k100 R_npv_large ({SCEN_TOKEN[sched]}) -- "
        "investigate before ratifying the comparators")
    _idx_nb = SELECTED_LARGE[sched]["idx"]
    _idx_z = int(np.argsort(_z["R_npv_large"], kind="stable")[int(np.ceil(0.50*(N_DRAWS-1)))])
    if _idx_z != _idx_nb:
        _rel = abs(float(_z["R_npv_large"][_idx_z]) - SELECTED_LARGE[sched]["score"]) \
            / abs(SELECTED_LARGE[sched]["score"])
        assert _rel <= 1e-9, (sched, _idx_nb, _idx_z)
        print(f"  NOTE ({SCEN_TOKEN[sched]}): near-tie rank swap vs k100 (rel {_rel:.1e}); "
              "in-notebook selection is authoritative")
    _k100_checked += 1
if _k100_checked:
    print(f"k100 pin: {_k100_checked}/6 schedules match tech_comparison's R_npv_large "
          "(worlds rtol 1e-9, scores rtol 1e-6, P50 rank identical or near-tie)")

print("\nSelected large100 mid-case draws (registered to selected_draws_large.csv):")
print(sel_large_df[["draw_index", "score", "pctile_in_MC"]].to_string())


k100 pin: 6/6 schedules match tech_comparison's R_npv_large (worlds rtol 1e-9, scores rtol 1e-6, P50 rank identical or near-tie)

Selected large100 mid-case draws (registered to selected_draws_large.csv):
                                  draw_index         score  pctile_in_MC
schedule              percentile                                        
EIA 2026 AEO high     p50               4588  1.591402e+05          50.0
Abou-Jaoude mod       p50               6297  2.172665e+05          50.0
IAEA high             p50               5712  5.291528e+05          50.0
McKinsey GEP 2025     p50               7176  6.676396e+05          50.0
COP28 tripling pledge p50               8155  1.215551e+06          50.0
2025 EO               p50               9427  1.818749e+06          50.0


In [14]:
# --- P5/P50/P95 of the pure-large ranking; P50 pinned to Step 3 ---
SELECTED_LARGE_PCT = {}
_sel_rows_lpct = []
for sched in SCHED_ORDER:
    order = np.argsort(mc_score_large[sched], kind="stable")
    for tag, q in PCT_TAGS.items():
        idx = int(order[int(np.ceil(q * (N_DRAWS - 1)))])
        SELECTED_LARGE_PCT[(sched, tag)] = {"idx": idx,
                                            "score": float(mc_score_large[sched][idx])}
        _sel_rows_lpct.append({"schedule": sched, "percentile": tag, "draw_index": idx,
                               "score": float(mc_score_large[sched][idx]),
                               "pctile_in_MC": round(100*float(
                                   (mc_score_large[sched] < mc_score_large[sched][idx]).mean()), 3),
                               **{c: float(WORLDS[sched].iloc[idx][c])
                                  for c in WORLDS[sched].columns}})
    # the re-derived P50 must equal the ported cell's (and therefore Step 3's) selection
    assert SELECTED_LARGE_PCT[(sched, "p50")]["idx"] == SELECTED_LARGE[sched]["idx"], sched
sel_large_pct_df = pd.DataFrame(_sel_rows_lpct).set_index(["schedule", "percentile"])
# overwrites the ported cell's p50-only file with the full three-percentile registration
sel_large_pct_df.to_csv(EXPORTS / "selected_draws_large.csv")

# Step-3 hard pin (ratified 2026-08-10) + frozen-file identity for the p50 rows
_LARGE100_PIN = {"eia_aeo_high": 4588, "abou_jaoude": 6297, "iaea_high": 5712,
                 "mckinsey": 7176, "cop28": 8155, "eo2025": 9427}
if N_DRAWS == 10000:
    for sched in SCHED_ORDER:
        assert SELECTED_LARGE_PCT[(sched, "p50")]["idx"] == _LARGE100_PIN[SCEN_TOKEN[sched]], sched
_frozen_lg = pd.read_csv(NB_DIR / "exports" / "smr100" / "selected_draws_large.csv",
                         index_col=["schedule", "percentile"])
for sched in SCHED_ORDER:
    assert int(_frozen_lg.loc[(sched, "p50"), "draw_index"])         == SELECTED_LARGE_PCT[(sched, "p50")]["idx"], sched
    assert np.isclose(float(_frozen_lg.loc[(sched, "p50"), "score"]),
                      SELECTED_LARGE_PCT[(sched, "p50")]["score"], rtol=1e-9), sched

# draw-identity assert vs mc_perdraw.npz for every selected large draw (as the smr cell does)
if _npz_sel_path.exists():
    _npz_lg = np.load(_npz_sel_path, allow_pickle=False)
    _lg_cols = list(_npz_lg["world_columns"].astype(str))
    _lg_checked = 0
    for (sched, tag), s in SELECTED_LARGE_PCT.items():
        tok = SCEN_TOKEN[sched]
        if _npz_lg[f"worlds_{tok}"].shape[0] != N_DRAWS:
            break
        assert _lg_cols == list(WORLDS[sched].columns)
        assert np.allclose(_npz_lg[f"worlds_{tok}"][s["idx"]],
                           WORLDS[sched].iloc[s["idx"]].to_numpy(dtype=float),
                           rtol=1e-9, atol=0), (sched, tag)
        _lg_checked += 1
    if _lg_checked:
        print(f"large-draw identity vs mc_perdraw.npz: {_lg_checked} selections match")

print("\nSelected large100 percentile draws (p50 = the Step 3 runs, pinned):")
print(sel_large_pct_df[["draw_index", "score", "pctile_in_MC"]].to_string())


large-draw identity vs mc_perdraw.npz: 18 selections match

Selected large100 percentile draws (p50 = the Step 3 runs, pinned):
                                  draw_index         score  pctile_in_MC
schedule              percentile                                        
EIA 2026 AEO high     p05               4046  1.286018e+05           5.0
                      p50               4588  1.591402e+05          50.0
                      p95                740  1.917481e+05          95.0
Abou-Jaoude mod       p05               1381  1.728268e+05           5.0
                      p50               6297  2.172665e+05          50.0
                      p95               9667  2.633730e+05          95.0
IAEA high             p05               2965  4.138293e+05           5.0
                      p50               5712  5.291528e+05          50.0
                      p95               3422  6.486382e+05          95.0
McKinsey GEP 2025     p05                382  5.167793e+05           

In [15]:
# --- The 18 large100 percentile records; input files for the 12 NEW cases only ---
CASES_LARGE_PCT = {}
for sched in SCHED_ORDER:
    tok = SCEN_TOKEN[sched]
    for tag in ("p05", "p50", "p95"):
        case = f"large100_{SHORT[tok]}_{tag}"
        idx = SELECTED_LARGE_PCT[(sched, tag)]["idx"]
        w1 = WORLDS[sched].iloc[[idx]].reset_index(drop=True)
        CASES_LARGE_PCT[case] = build_case_rec(case, sched, tag, w1,
                                               SELECTED_LARGE_PCT[(sched, tag)]["score"],
                                               {"draw_index": idx}, program="large",
                                               score_dist=mc_score_large[sched])

# p50 record continuity vs the frozen Step-1 registration (the Step 3 runs' inputs)
_frozen_lgc = pd.read_csv(NB_DIR / "exports" / "smr100" / "selected_cases_large.csv",
                          index_col="case")
for sched in SCHED_ORDER:
    case = f"large100_{SHORT[SCEN_TOKEN[sched]]}_p50"
    assert int(_frozen_lgc.loc[case, "draw_index"]) == CASES_LARGE_PCT[case]["draw_index"], case
    assert np.isclose(float(_frozen_lgc.loc[case, "score"]),
                      CASES_LARGE_PCT[case]["score"], rtol=1e-9), case
    assert float(_frozen_lgc.loc[case, "occ_large_2050"])         == CASES_LARGE_PCT[case]["occ_large_2050"], case

selected_cases_large_pct = pd.DataFrame(
    {c: {k: v for k, v in rec.items() if not isinstance(v, np.ndarray)}
     for c, rec in CASES_LARGE_PCT.items()}).T.set_index("case")
selected_cases_large_pct.to_csv(EXPORTS / "selected_cases_large.csv")

# Only the P5/P95 cases are new runs -- the export cells below write files for these alone.
NEW_CASES = {c: r for c, r in CASES_LARGE_PCT.items() if r["case_type"] in ("p05", "p95")}
ALL_CASES = NEW_CASES        # the ported Export 1/2 cells iterate ALL_CASES
print(f"built {len(CASES_LARGE_PCT)} large100 percentile records "
      f"(p50 continuity vs Step 3 asserted); {len(NEW_CASES)} new cases get input files")


built 18 large100 percentile records (p50 continuity vs Step 3 asserted); 12 new cases get input files


## Exports to ReEDS

The Export 1/2 cells are ported verbatim; with `ALL_CASES` bound to the 12 new large100
percentile cases they write exactly 24 plantchar files, 12 `financials_tech_mc_*`, 12
`construction_times_mc_*`, the idempotent `dollaryear.csv` registration, and re-write the
shared `construction_schedules_mc.csv` byte-identically (guard-verified). The new cases
matrix is then assembled by the Step-4 assembler below.


In [16]:
# Export 1: plant-characteristics files + dollaryear registration
PLANTCHAR_DIR = REPO_ROOT / "inputs" / "plant_characteristics"
REEDS_TECH_NAME = {"large": "Nuclear", "smr": "Nuclear-SMR"}
ATB_FILE = {"large": "nuclear_ATB_2024_moderate.csv", "smr": "nuclear-smr_ATB_2024_moderate.csv"}
atb_base = {t: pd.read_csv(PLANTCHAR_DIR / ATB_FILE[t]) for t in TECH}

def plantchar_name(tech, case):
    return f"{'nuclear' if tech == 'large' else 'nuclear-smr'}_mc_{case}"

written_plantchar = []
for case, info in ALL_CASES.items():
    for tech in TECH:
        occ = info[f"occ_{tech}"]                             # (T,) 2024-2050, 2022 $/kW
        fom_d, vom_d = info[f"fom_{tech}"], info[f"vom_{tech}"]
        df = atb_base[tech].copy()
        for t_i, y in enumerate(YEARS):
            if y >= ANCHOR:                                   # pre-anchor years keep ATB values
                df.loc[df["t"] == y, "capcost"] = round(float(occ[t_i]), 1)
                df.loc[df["t"] == y, "fom"] = fom_d   # case-consistent O&M (2026-08-03)
                df.loc[df["t"] == y, "vom"] = vom_d
        name = plantchar_name(tech, case)
        df.to_csv(PLANTCHAR_DIR / f"{name}.csv", index=False)
        written_plantchar.append(name)

doll_path = PLANTCHAR_DIR / "dollaryear.csv"
doll = pd.read_csv(doll_path)
new_rows = [{"Scenario": n, "Dollar.Year": 2022} for n in written_plantchar
            if n not in set(doll["Scenario"])]
if new_rows:
    doll = pd.concat([doll, pd.DataFrame(new_rows)], ignore_index=True)
    doll.to_csv(doll_path, index=False)
print(f"wrote {len(written_plantchar)} plant-characteristics files; "
      f"registered {len(new_rows)} new dollaryear rows (idempotent)")

wrote 24 plant-characteristics files; registered 0 new dollaryear rows (idempotent)


In [17]:
# Export 2: shared construction-schedules columns + per-case financials_tech / construction_times
cs_mc = pd.read_csv(FIN_DIR / "construction_schedules_default.csv")
for tech, prefix in [("large", "NL"), ("smr", "NS")]:
    for n in range(1, 11):
        x = _resample_schedule(FIN["sched"][tech], n)
        col = np.zeros(len(cs_mc))
        col[1:n+1] = x                        # row 0 is the 'NA' (exponent-0) row
        cs_mc[f"{prefix}{n}"] = np.round(col, 6)
cs_mc.to_csv(FIN_DIR / "construction_schedules_mc.csv", index=False)

def dur_to_years(dur_row):
    """Designed duration path (months, (T,)) -> ReEDS spend-years labels per year."""
    return np.clip(np.round(np.asarray(dur_row) / 12.0).astype(int), 1, 10)

ft_base = pd.read_csv(FIN_DIR / "financials_tech_ATB2024.csv")
ct_base = pd.read_csv(FIN_DIR / "construction_times_default.csv")
SCH_PREFIX = {"large": "NL", "smr": "NS"}

for case, info in ALL_CASES.items():
    ft = ft_base.copy()
    ft["construction_sch"] = ft["construction_sch"].astype(str)
    ct = ct_base.copy()
    for tech in TECH:
        n_years = dur_to_years(info[f"dur_{tech}"])                     # (T,)
        iname = REEDS_TECH_NAME[tech]
        for t_i, y in enumerate(YEARS):
            if y >= ANCHOR:
                mask = (ft["i"] == iname) & (ft["t"] == y)
                ft.loc[mask, "construction_sch"] = f"{SCH_PREFIX[tech]}{n_years[t_i]}"
                ct.loc[(ct["i"] == iname) & (ct["t_online"] == y),
                       "construction_time"] = int(n_years[t_i])
    ft.to_csv(FIN_DIR / f"financials_tech_mc_{case}.csv", index=False)
    ct.to_csv(FIN_DIR / f"construction_times_mc_{case}.csv", index=False)
print(f"wrote construction_schedules_mc.csv (+20 profile columns) and "
      f"{len(ALL_CASES)} financials_tech_mc_* + {len(ALL_CASES)} construction_times_mc_* files")

wrote construction_schedules_mc.csv (+20 profile columns) and 12 financials_tech_mc_* + 12 construction_times_mc_* files


In [18]:
# --- The Step 4 sensitivity definitions: single source of truth (embedded in metadata) ---
# Current-switch-name translations of the NREL Standard Scenarios sensitivities.
# cases_standardscenarios.csv in this fork is STALE (dead switch names: convscen, upvscen,
# ...) and will not parse -- do not copy it. Values verified against cases.csv Choices and
# against the input tree (existence checks below).
#   gaslo/gashi: AEO 2026 High/Low Oil & Gas supply (HOG = LOW gas price, LOG = HIGH).
#   demhi: EER2025 family required -- EER2023 hard-errors against the default
#          resource_adequacy_years; demand_EER2025_100by2050.h5 is fetched from Zenodo via
#          inputs/remote_files.csv (one heads-up line in the email to NREL).
#   relo:  full StdScen Adv_RE fidelity (incl. hydro low, geodiscov TI, distpv low-RE-cost).
#   rehi:  StdScen Con_RE (no hydro/geodiscov override there -- no hydro 'high' exists).
#   translim: StdScen Limited_Trans verbatim. P7 caveat applies: regional/transmission
#          readouts across cost worlds are siting-degenerate -- read national aggregates.
# (An "int" intertemporal arm was defined here 2026-08-12 and removed 2026-08-13: NREL
#  confirmed the intertemporal solver has not worked for years.)
SENS = {
    "gaslo":    {"ngscen": "AEO_2026_HOG"},
    "gashi":    {"ngscen": "AEO_2026_LOG"},
    "demhi":    {"GSw_LoadProfiles": "EER2025_100by2050"},
    "relo":     {"plantchar_upv": "upv_ATB_2024_advanced",
                 "plantchar_onswind": "ons-wind_ATB_2024_advanced",
                 "plantchar_ofswind": "ofs-wind_ATB_2024_advanced",
                 "plantchar_battery": "battery_ATB_2024_advanced",
                 "plantchar_csp": "csp_ATB_2024_advanced",
                 "plantchar_geo": "geo_ATB_2024_advanced",
                 "plantchar_hydro": "hydro_ATB_2019_low",
                 "geodiscov": "TI",
                 "distpvscen": "stscen2023_lowre"},
    "rehi":     {"plantchar_upv": "upv_ATB_2024_conservative",
                 "plantchar_onswind": "ons-wind_ATB_2024_conservative",
                 "plantchar_ofswind": "ofs-wind_ATB_2024_conservative",
                 "plantchar_battery": "battery_ATB_2024_conservative",
                 "plantchar_csp": "csp_ATB_2024_conservative",
                 "plantchar_geo": "geo_ATB_2024_conservative",
                 "distpvscen": "stscen2023_highre"},
    "translim": {"GSw_TransRestrict": "transreg", "GSw_TransInvMaxLongTerm": "1.07"},
}
SENS_ORDER = ["gaslo", "gashi", "demhi", "relo", "rehi", "translim"]
# matrix rows added beyond the Step 3 set (blank Default cells inherit root cases.csv
# defaults)
SENS_SWITCH_ROWS = ["ngscen", "GSw_LoadProfiles", "plantchar_upv", "plantchar_onswind",
                    "plantchar_ofswind", "plantchar_battery", "plantchar_csp",
                    "plantchar_geo", "plantchar_hydro", "geodiscov", "distpvscen",
                    "GSw_TransRestrict", "GSw_TransInvMaxLongTerm"]
assert {k for ov in SENS.values() for k in ov} <= set(SENS_SWITCH_ROWS)

# referenced-input existence: every value must resolve to real files in this fork
for scen in ("AEO_2026_HOG", "AEO_2026_LOG"):
    for stem in ("alpha", "ng", "ng_demand", "ng_tot_demand"):
        assert (REPO_ROOT / "inputs" / "fuelprices" / f"{stem}_{scen}.csv").exists(), (stem, scen)
for ov in SENS.values():
    for row, val in ov.items():
        if row.startswith("plantchar_"):
            assert (REPO_ROOT / "inputs" / "plant_characteristics" / f"{val}.csv").exists(), val
        if row == "distpvscen":
            assert (REPO_ROOT / "inputs" / "dgen_model_inputs" / val).is_dir(), val
_remote = (REPO_ROOT / "inputs" / "remote_files.csv").read_text()
assert "demand_EER2025_100by2050.h5" in _remote, "demand h5 not in remote_files.csv"
print(f"SENS defined: {len(SENS)} sensitivity arms; all referenced input files exist "
      "(demand h5 via Zenodo remote_files entry)")


SENS defined: 6 sensitivity arms; all referenced input files exist (demand h5 via Zenodo remote_files entry)


In [19]:
# --- Export 3 (Step 4): the 120-column cases matrix ---
# Same row schema as smr100_case_export's cases_matrix, extended with the sensitivity rows.
# smr100 sensitivity columns are pure FILE-POINTER COPIES of their Step 3 base case
# (the eq-flip mechanism) plus their switch overrides; large100 p05/p95 columns carry
# their own new files and the per-case large-only mandate.
STEP4_CASES = {}
for sens in SENS_ORDER:
    for sched in SCHED_ORDER:
        stem = SHORT[SCEN_TOKEN[sched]]
        for tag in ("p05", "p50", "p95"):
            base = f"smr100_{stem}_{tag}"
            STEP4_CASES[f"{base}_{sens}"] = {
                "scen_token": SCEN_TOKEN[sched], "winner": "smr", "file_case": base,
                "base_case": base, "sens": sens, "overrides": dict(SENS[sens])}
for sched in SCHED_ORDER:
    stem = SHORT[SCEN_TOKEN[sched]]
    for tag in ("p05", "p95"):
        case = f"large100_{stem}_{tag}"
        rec = CASES_LARGE_PCT[case]
        STEP4_CASES[case] = {"scen_token": rec["scen_token"], "winner": "large",
                             "overrides": {}}
assert len(STEP4_CASES) == 6*18 + 12 == 120

_YEARSET = "2010_2015_2020_2023_2026_2029_2031_2032_2033_2034_2035_2038_2041_2044_2047_2050"

def cases_matrix_step4(case_dict):
    n = len(case_dict)
    def _prog(c):
        return case_dict[c].get("winner", "smr")
    def _fc(c):
        return case_dict[c].get("file_case", c)
    def _ov(c, row):
        return case_dict[c].get("overrides", {}).get(row, "")
    rows = {
        "ignore": ["0"] + [""] * n,
        "timetype": ["seq"] + [""] * n,
        "yearset": [_YEARSET] + [""] * n,
        "endyear": ["2050"] + [""] * n,
        "GSw_NuclearCapMandate": ["1"] + [""] * n,          # floor everywhere (issue 6)
        "GSw_NuclearCapMandate_Scale": ["1"] + [""] * n,
        "GSw_NuclearCapMandateTechScen": ["smr"] + ["" if _prog(c) == "smr" else "large"
                                                    for c in case_dict],
        "GSw_NuclearLearning": ["0"] + [""] * n,            # exogenous costs by design
        "incentives_suffix": ["obbba_nonuclearitc"] + [""] * n,   # no-nuclear-ITC baseline
        "construction_schedules_suffix": ["mc"] + [""] * n,
        **{row: [""] + [_ov(c, row) for c in case_dict] for row in SENS_SWITCH_ROWS},
        "GSw_NuclearCapMandateScen": [""] + [case_dict[c]["scen_token"]
                                             + ("_smr" if _prog(c) == "smr" else "_large")
                                             for c in case_dict],
        "plantchar_nuclear": [""] + [plantchar_name("large", _fc(c)) for c in case_dict],
        "plantchar_nuclear_smr": [""] + [plantchar_name("smr", _fc(c)) for c in case_dict],
        "financials_tech_suffix": [""] + [f"mc_{_fc(c)}" for c in case_dict],
        "construction_times_suffix": [""] + [f"mc_{_fc(c)}" for c in case_dict],
    }
    return pd.DataFrame(rows, index=["Default Value"] + list(case_dict)).T

cases_step4 = cases_matrix_step4(STEP4_CASES)
STEP4_CASES_PATH = REPO_ROOT / "cases_nuclearlearning_step4.csv"
cases_step4.to_csv(STEP4_CASES_PATH, index_label="")
print(f"wrote {STEP4_CASES_PATH.name}: {len(STEP4_CASES)} cases "
      f"(launch: python runreeds.py -b <batch> -c nuclearlearning_step4)")
cases_step4.head(12)


wrote cases_nuclearlearning_step4.csv: 120 cases (launch: python runreeds.py -b <batch> -c nuclearlearning_step4)


,Default Value,smr100_eia_p05_gaslo,smr100_eia_p50_gaslo,smr100_eia_p95_gaslo,smr100_aj_p05_gaslo,smr100_aj_p50_gaslo,smr100_aj_p95_gaslo,smr100_iaea_p05_gaslo,smr100_iaea_p50_gaslo,smr100_iaea_p95_gaslo,...,large100_aj_p05,large100_aj_p95,large100_iaea_p05,large100_iaea_p95,large100_mck_p05,large100_mck_p95,large100_cop28_p05,large100_cop28_p95,large100_eo_p05,large100_eo_p95
ignore,0,,,,,,,,,,...,,,,,,,,,,
timetype,seq,,,,,,,,,,...,,,,,,,,,,
yearset,2010_2015_2020_2023_2026_2029_2031_2032_2033_2...,,,,,,,,,,...,,,,,,,,,,
endyear,2050,,,,,,,,,,...,,,,,,,,,,
GSw_NuclearCapMandate,1,,,,,,,,,,...,,,,,,,,,,
GSw_NuclearCapMandate_Scale,1,,,,,,,,,,...,,,,,,,,,,
GSw_NuclearCapMandateTechScen,smr,,,,,,,,,,...,large,large,large,large,large,large,large,large,large,large
GSw_NuclearLearning,0,,,,,,,,,,...,,,,,,,,,,
incentives_suffix,obbba_nonuclearitc,,,,,,,,,,...,,,,,,,,,,
construction_schedules_suffix,mc,,,,,,,,,,...,,,,,,,,,,


## QA suite

Ported identity checks (QA-0..QA-3) plus the Step-4-specific gates: large100 percentile
identity/placement (QA-S4), export round-trips + `cases.csv` validation (QA-S5), the
matrix regression guard incl. pointer identity against the Step 3 file on disk and the
Step-3-artifact byte-identity snapshot (QA-S6), and baseline/input existence (QA-S7).


In [20]:
# QA-0 — same sample space as the companion: if mc_cost_trajectories.ipynb's per-draw export
# (exports/mc_perdraw.npz) is present and was generated at the same draw count, this
# notebook's drawn worlds must match it draw-for-draw (same code, same seed streams), and the
# SMR full-program trajectory arrays must match too. Tolerance is floating-point rounding
# only (rtol 1e-9): the companion runs on a different kernel/BLAS build, which perturbs the
# copula matmul in the last bit (~2e-16 relative) — the underlying random stream is identical
# (the plain-rng columns like `s` match bit-for-bit). Skips gracefully if absent/stale.
_npz_path = NB_DIR / "exports" / "mc_perdraw.npz"
if _npz_path.exists():
    _pd_npz = np.load(_npz_path, allow_pickle=False)
    _tok0 = SCEN_TOKEN[SCHED_ORDER[0]]
    if _pd_npz[f"worlds_{_tok0}"].shape[0] != N_DRAWS:
        print(f"QA-0 SKIPPED: mc_perdraw.npz has {_pd_npz[f'worlds_{_tok0}'].shape[0]} draws, "
              f"this run has {N_DRAWS} — re-run the companion notebook to compare")
    else:
        cols = list(_pd_npz["world_columns"].astype(str))
        for sched in SCHED_ORDER:
            tok = SCEN_TOKEN[sched]
            assert cols == list(WORLDS[sched].columns), "world column layout diverged"
            ours_w = WORLDS[sched].to_numpy(dtype=float)
            assert np.allclose(_pd_npz[f"worlds_{tok}"], ours_w, rtol=1e-9, atol=0), tok
            assert np.array_equal(_pd_npz[f"worlds_{tok}"][:, cols.index("s")],
                                  WORLDS[sched]["s"].to_numpy()), (tok, "stream identity")
            for ch, ours in (("occ", "occ_smr"), ("dur", "dur_smr"),
                             ("ccmult", "ccmult_smr"), ("fincapex", "fincapex_smr")):
                assert np.allclose(_pd_npz[f"{ch}_{tok}_smr"], results[sched][ours],
                                   rtol=1e-9, atol=0), (tok, ch)
        print("QA-0 PASSED: drawn worlds and SMR full-program trajectories match the companion "
              "notebook's mc_perdraw.npz draw-for-draw (bit-identical random stream; "
              "differences bounded by cross-kernel floating-point rounding) — "
              "same sample space by construction")
else:
    print("QA-0 SKIPPED: exports/mc_perdraw.npz not found (run the companion notebook to enable "
          "the sample-space cross-check)")

QA-0 PASSED: drawn worlds and SMR full-program trajectories match the companion notebook's mc_perdraw.npz draw-for-draw (bit-identical random stream; differences bounded by cross-kernel floating-point rounding) — same sample space by construction


In [21]:
# QA-1 — the comonotone draw implies the honest orderings on every draw (the SMR range starts
# higher and ends higher, so same-percentile draws always have the SMR pricier at the anchor
# and learning at least as fast); marginals inside published ranges.
for sched in SCHED_ORDER:
    w = WORLDS[sched]
    assert (w["boak_smr"] > w["boak_large"]).all(), sched
    assert (w["lr_smr"] >= w["lr_large"]).all(), sched
    for tech in TECH:
        t = TECH[tech]
        assert w[f"boak_{tech}"].between(t["boak_lo"], t["boak_hi"]).all(), (sched, tech)
        assert w[f"lr_{tech}"].between(t["lr_lo"], t["lr_hi"]).all(), (sched, tech)
print("QA-1 PASSED: boak_smr > boak_large and lr_smr >= lr_large on every draw "
      "(implied by the comonotone draw); all marginals inside their published ranges")

QA-1 PASSED: boak_smr > boak_large and lr_smr >= lr_large on every draw (implied by the comonotone draw); all marginals inside their published ranges


In [22]:
# QA-2 — anchor identity: OCC(2030) == the drawn anchor cost, every draw, both channels.
for sched in SCHED_ORDER:
    w = WORLDS[sched]
    for tech in TECH:
        assert np.allclose(results[sched][f"occ_{tech}"][:, yi(ANCHOR)],
                           w[f"boak_{tech}"].values), (sched, tech)
print("QA-2 PASSED: every draw hits its anchor cost exactly at 2030 "
      "(SMR full-program channel and large international-only channel)")

QA-2 PASSED: every draw hits its anchor cost exactly at 2030 (SMR full-program channel and large international-only channel)


In [23]:
# QA-3 — financing regression pin + the large counterfactual behaves as designed:
# with zero US large builds its OCC declines through international spillover plus the
# drawn cross-tech fraction x of the SMR program (companion S5 Step 5').
_ccm = ccmult_from_duration(72.0, 1.08, FIN["sched"]["large"])
assert abs(_ccm - 1.268124) < 1e-5, _ccm
_decline = 1.0 - results[SCHED_ORDER[-1]]["occ_large"][:, yi(2050)] \
                 / results[SCHED_ORDER[-1]]["occ_large"][:, yi(ANCHOR)]
# Bound history: 0.40 pre-2026-08-06 (intl-only, no cross-tech term); 0.65 since the
# cross-tech x adoption (kept through the same-day routing revert): the loser receives
# x_sl * the 1011-unit SMR program in the cross-firm channel, and at the extreme corner
# (s~1, u~1, x_sl~0.3, m=4, tiny-base, rho=1) the additively-pooled cross-firm stock
# alone drives a ~50%+ decline. Physics, not a bug; 65% still flags a runaway channel.
assert _decline.max() < 0.65, "large loser channel declining implausibly fast"
print(f"QA-3 PASSED: ccmult pin 1.268124 reproduced; large-reactor counterfactual OCC declines "
      f"{np.percentile(_decline, 50):.1%} (median) to {_decline.max():.1%} (max) by 2050 — "
      "loser channel (intl spillover + x * SMR program), as designed")

QA-3 PASSED: ccmult pin 1.268124 reproduced; large-reactor counterfactual OCC declines 3.4% (median) to 53.6% (max) by 2050 — loser channel (intl spillover + x * SMR program), as designed


In [24]:
# QA-S4 -- large100 percentile cases: draw identity, placement, ordering, anchor identity.
NOMINAL_PCT = {"p05": 5.0, "p50": 50.0, "p95": 95.0}
for case, info in CASES_LARGE_PCT.items():
    sched, idx = info["schedule"], info["draw_index"]
    row = WORLDS[sched].iloc[idx]
    for k_rec, k_w in (("lr_large", "lr_large"), ("lr_smr", "lr_smr"),
                       ("boak_large", "boak_large"), ("boak_smr", "boak_smr"),
                       ("u", "u"), ("s_spill", "s"), ("x_ls", "x_ls"), ("x_sl", "x_sl"),
                       ("n_vendors", "n_vendors"), ("ces_rho", "ces_rho"),
                       ("dur_lambda", "dur_lambda"), ("dur_z", "dur_z")):
        assert abs(float(info[k_rec]) - float(row[k_w])) < 1e-12, (case, k_rec)
    pct = 100.0*float((mc_score_large[sched] < info["score"]).mean())
    assert abs(pct - NOMINAL_PCT[info["case_type"]]) <= 0.5, (case, pct)
    assert info["winner"] == "large", case
    # role-swap anchor identities on the exported paths
    assert abs(info["occ_large"][yi(ANCHOR)] - info["boak_large"]) < 0.5, case
    assert abs(info["occ_smr"][yi(ANCHOR)] - info["boak_smr"]) < 0.5, case
for sched in SCHED_ORDER:
    sc = {t: SELECTED_LARGE_PCT[(sched, t)]["score"] for t in NOMINAL_PCT}
    assert sc["p05"] < sc["p50"] < sc["p95"], sched
print("QA-S4 PASSED: all 18 large100 percentile records identical to their registered "
      "draws, at nominal rank +/-0.5 pctile, strictly ordered, anchors exact")


QA-S4 PASSED: all 18 large100 percentile records identical to their registered draws, at nominal rank +/-0.5 pctile, strictly ordered, anchors exact


In [25]:
# QA-S5 -- export round-trips for the 12 new cases + the new cases file vs cases.csv.
import re as _re

# (a) plantchar round-trip: capcost 2030-2050 = the case's OCC path; pre-anchor = ATB;
#     case-consistent fom/vom
for case, info in NEW_CASES.items():
    for tech in TECH:
        back = pd.read_csv(PLANTCHAR_DIR / f"{plantchar_name(tech, case)}.csv").set_index("t")
        occ = info[f"occ_{tech}"]
        for y in (2030, 2040, 2050):
            assert abs(back.loc[y, "capcost"] - occ[yi(y)]) < 0.1, (case, tech, y)
            assert abs(back.loc[y, "fom"] - info[f"fom_{tech}"]) < 1e-6, (case, tech, y)
            assert abs(back.loc[y, "vom"] - info[f"vom_{tech}"]) < 1e-6, (case, tech, y)
        _atb_t = atb_base[tech].set_index("t")
        assert abs(back.loc[2025, "capcost"] - _atb_t.loc[2025, "capcost"]) < 1e-6
        assert abs(back.loc[2025, "fom"] - _atb_t.loc[2025, "fom"]) < 1e-6

# (b) ccmult from the WRITTEN schedule labels equals the notebook's ccmult (all 12 cases)
cs_back = pd.read_csv(FIN_DIR / "construction_schedules_mc.csv")
for case, info in NEW_CASES.items():
    ft_back = pd.read_csv(FIN_DIR / f"financials_tech_mc_{case}.csv")
    for tech in TECH:
        iname = REEDS_TECH_NAME[tech]
        for y in (2030, 2040, 2050):
            label = ft_back.loc[(ft_back["i"] == iname) & (ft_back["t"] == y),
                                "construction_sch"].iloc[0]
            frac = pd.to_numeric(cs_back[label], errors="coerce").fillna(0.0).to_numpy()
            frac = frac[frac > 0]
            exps = np.arange(len(frac)) + 0.5
            ccm_file = 1.0 + float(np.sum(frac/frac.sum()
                                          * (FIN["interest_base"][yi(y)]**exps - 1.0)))
            mo = info[f"dur_{tech}"][yi(y)]
            ccm_nb = ccmult_from_duration(mo, FIN["interest_base"][yi(y)], FIN["sched"][tech])
            assert abs(ccm_file - ccm_nb) < 2e-5, (case, tech, y, ccm_file, ccm_nb)

# (c) the Step 4 cases file validates against cases.csv's own index and Choices regexes
cases_master = pd.read_csv(REPO_ROOT / "cases.csv", index_col=0)
cases_back = pd.read_csv(STEP4_CASES_PATH, index_col=0, dtype=str).fillna("")
for switch in cases_back.index:
    if switch == "ignore":
        continue
    assert switch in cases_master.index, f"unknown switch {switch}"
    choices = str(cases_master.loc[switch, "Choices"])
    for col in cases_back.columns:
        val = cases_back.loc[switch, col]
        if val == "":
            continue
        if choices in ("N/A", "nan", "int", "float"):
            continue
        tokens = [c.strip() for c in _re.split("[;,]", choices)]
        assert any(_re.match(tok + "$", str(val)) for tok in tokens), (switch, val, choices)

# (d) dollaryear registration idempotent and complete for the new plantchar files
doll_back = pd.read_csv(doll_path)
for n in written_plantchar:
    assert (doll_back["Scenario"] == n).sum() == 1, n

# (e) the fleet-inclusive {scen}_large trajectories the large100 columns point at:
#     existing 80-yr fleet path + this notebook's own program additions
_fleet_gw = US_GW_2024 - np.cumsum(US_RET)
_nl_dir = REPO_ROOT / "inputs" / "nuclear_learning"
for sched in SCHED_ORDER:
    _trajs = {}
    for b in ("large", "smr"):
        tr = pd.read_csv(_nl_dir / f"nuclear_cap_trajectory_{SCEN_TOKEN[sched]}_{b}.csv",
                         comment="*", header=None, names=["t", "mw"]).set_index("t")
        _trajs[b] = tr["mw"].reindex(YEARS).to_numpy() / 1000.0
    assert np.allclose(_trajs["large"], _fleet_gw + np.cumsum(GW_ADD[sched]), atol=2e-3), sched
    assert np.allclose(_trajs["large"] - _trajs["smr"], _fleet_gw, atol=4e-3), sched

print("QA-S5 PASSED: plantchar + ccmult round-trips on all 12 new cases; the Step 4 cases "
      "file validates against cases.csv Choices; dollaryear idempotent; fleet-inclusive "
      "large trajectory identity holds")


QA-S5 PASSED: plantchar + ccmult round-trips on all 12 new cases; the Step 4 cases file validates against cases.csv Choices; dollaryear idempotent; fleet-inclusive large trajectory identity holds


In [26]:
# QA-S6 -- matrix regression guard + Step 3 byte-identity.
# (a) exact column order: 6 market sens x 18, then large100 p05/p95 x 12
_expected_cols = ["Default Value"]
for sens in SENS_ORDER:
    for sched in SCHED_ORDER:
        for tag in ("p05", "p50", "p95"):
            _expected_cols.append(f"smr100_{SHORT[SCEN_TOKEN[sched]]}_{tag}_{sens}")
for sched in SCHED_ORDER:
    for tag in ("p05", "p95"):
        _expected_cols.append(f"large100_{SHORT[SCEN_TOKEN[sched]]}_{tag}")
assert list(cases_step4.columns) == _expected_cols
assert len(cases_step4.columns) == 1 + 120

# (b) pointer identity vs the Step 3 file ON DISK: every sensitivity column's five
#     pointer rows byte-equal its base case's column in cases_nuclearlearning_smr100.csv
_step3 = pd.read_csv(REPO_ROOT / "cases_nuclearlearning_smr100.csv",
                     index_col=0, dtype=str).fillna("")
_step4 = pd.read_csv(STEP4_CASES_PATH, index_col=0, dtype=str).fillna("")
PTR_ROWS = ["GSw_NuclearCapMandateScen", "plantchar_nuclear", "plantchar_nuclear_smr",
            "financials_tech_suffix", "construction_times_suffix"]
for c, rec in STEP4_CASES.items():
    if "base_case" not in rec:
        continue
    b = rec["base_case"]
    for r in PTR_ROWS:
        assert _step4.loc[r, c] == _step3.loc[r, b] != "", (c, r)

# (c) override audit: each sensitivity column sets exactly its SENS rows + the five pointers
for c, rec in STEP4_CASES.items():
    nonblank = set(_step4.index[_step4[c] != ""])
    if "base_case" in rec:
        assert nonblank == set(PTR_ROWS) | set(rec["overrides"]), (c, nonblank)
        for row, val in rec["overrides"].items():
            assert _step4.loc[row, c] == val, (c, row)
    else:   # large100 percentile columns: own files + the large-only mandate
        assert nonblank == set(PTR_ROWS) | {"GSw_NuclearCapMandateTechScen"}, (c, nonblank)
        assert _step4.loc["GSw_NuclearCapMandateTechScen", c] == "large", c
        assert _step4.loc["GSw_NuclearCapMandateScen", c].endswith("_large"), c
        for r in ("plantchar_nuclear", "financials_tech_suffix"):
            assert c in _step4.loc[r, c], (c, r)   # own file names, never a pointer copy

# (d) no collision with the Step 3 matrix; Default Value column consistent with it
assert not (set(_step4.columns) - {"Default Value"}) & (set(_step3.columns) - {"Default Value"})
for r in _step3.index:
    assert _step4.loc[r, "Default Value"] == _step3.loc[r, "Default Value"], r
for r in SENS_SWITCH_ROWS:
    assert _step4.loc[r, "Default Value"] == "", r

# (e) Step 3 artifacts byte-identical (incl. the deterministically re-written
#     construction_schedules_mc.csv)
for p, sha0 in GUARD_SHA.items():
    assert _sha(p) == sha0, f"Step 3 artifact changed: {p}"

print("QA-S6 PASSED: 120 columns in the registered order; every sensitivity column's "
      "pointers byte-equal its Step 3 base column on disk; overrides exact and minimal; "
      "no column collision; shared defaults identical; all snapshotted Step 3 artifacts "
      "byte-identical")


QA-S6 PASSED: 120 columns in the registered order; every sensitivity column's pointers byte-equal its Step 3 base column on disk; overrides exact and minimal; no column collision; shared defaults identical; all snapshotted Step 3 artifacts byte-identical


In [27]:
# QA-S7 -- baseline + referenced inputs.
# (a) the no-nuclear-ITC baseline rides as the shared default with blank per-case cells
assert _step4.loc["incentives_suffix", "Default Value"] == "obbba_nonuclearitc"
assert (_step4.loc["incentives_suffix", _step4.columns[1:]] == "").all()
# (b) floor mode everywhere; learning off; mc construction schedules
for r, v in (("GSw_NuclearCapMandate", "1"), ("GSw_NuclearLearning", "0"),
             ("construction_schedules_suffix", "mc")):
    assert _step4.loc[r, "Default Value"] == v
    assert (_step4.loc[r, _step4.columns[1:]] == "").all()
# (c) sequential solver everywhere (the int foresight arm was cancelled 2026-08-13 --
#     NREL: the intertemporal solver has not worked for years)
assert _step4.loc["timetype", "Default Value"] == "seq"
assert (_step4.loc["timetype", _step4.columns[1:]] == "").all()
assert not any(c.endswith("_int") for c in _step4.columns)
# (d) every sensitivity value's input file existence was asserted at definition time
#     (SENS cell); re-assert the two dgen dirs + the remote demand entry here for the record
for v in ("stscen2023_lowre", "stscen2023_highre"):
    assert (REPO_ROOT / "inputs" / "dgen_model_inputs" / v).is_dir(), v
assert "demand_EER2025_100by2050.h5" in (REPO_ROOT / "inputs" / "remote_files.csv").read_text()
print("QA-S7 PASSED: no-nuclear-ITC + floor + learning-off defaults carried; sequential "
      "solver everywhere; sensitivity inputs present (demand h5 via Zenodo)")


QA-S7 PASSED: no-nuclear-ITC + floor + learning-off defaults carried; sequential solver everywhere; sensitivity inputs present (demand h5 via Zenodo)


## Run metadata

In [28]:
import subprocess
from datetime import datetime, timezone

try:
    _git = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT,
                          capture_output=True, text=True).stdout.strip()
except Exception:
    _git = "unavailable"

meta = {
    "notebook": "step4_case_export.ipynb",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "fork_git_head": _git,
    "master_seed": MASTER_SEED,
    "built_by": "_build_step4_case_export.py (ported cells verbatim from "
                "smr100_case_export.ipynb with build-time content asserts)",
    "design": {
        "batch": "Step 4 sensitivities: 108 market-sensitivity columns (18 smr100 cases x "
                 "6 sensitivities, full coverage ratified 2026-08-12) + 12 large100 "
                 "percentile completions (p05/p95 of the pure-large ranking; p50 ran in "
                 "Step 3). smr100 columns are file-pointer copies of their Step 3 base "
                 "cases (the eq-flip mechanism) -- no new nuclear input files; only the "
                 "12 large100 cases carry new files.",
        "sensitivities": SENS,
        "run_arithmetic": {"step3_spent": 25, "step4_batch": len(STEP4_CASES),
                           "total": 25 + len(STEP4_CASES),
                           "ceiling": "145 (amended 2026-08-13 from 163 on cancellation "
                                      "of the int arm; 163 ratified 2026-08-12; "
                                      "previously 126 -> 133 on 2026-08-10)"},
        "issue7_invariant": "layered (ratified 2026-08-12): dual-decay (bridge) shape "
                            "primary, trajectory ranking secondary, subsidy level "
                            "tertiary -- reporting design, not a case-file input",
        "int_arm_cancelled": "the 18-column intertemporal (timetype=int) foresight arm "
                             "ratified 2026-08-12 was removed 2026-08-13: NREL confirmed "
                             "by email that the intertemporal solver has not worked for "
                             "years; the sequential/myopic caveat stays analytical",
        "selection_continuity": "smr100 selections asserted identical to "
                                "exports/smr100/selected_draws.csv; large100 p50 pinned to "
                                "the Step 3 hard pin + frozen registrations",
        "large100_pctile_draw_indices": {c: int(r["draw_index"])
                                         for c, r in CASES_LARGE_PCT.items()},
        "copula_set": COPULA_SET, "n_draws": N_DRAWS, "ranking": RANKING_FUNCTIONAL},
    "cases_sens": [c for c, r in STEP4_CASES.items() if "base_case" in r],
    "cases_large_pctile": [c for c in STEP4_CASES if c.startswith("large100")],
    "reeds_files": {"cases_file": STEP4_CASES_PATH.name,
                    "plantchar": sorted(written_plantchar),
                    "financials_tech": [f"financials_tech_mc_{c}.csv" for c in NEW_CASES],
                    "construction_times": [f"construction_times_mc_{c}.csv"
                                           for c in NEW_CASES],
                    "construction_schedules": "construction_schedules_mc.csv (unchanged, "
                                              "byte-identity asserted)"},
    "exports": {p.name: _sha(p)[:16] for p in sorted(EXPORTS.glob("*.csv"))},
}
with open(EXPORTS / "step4_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps({k: meta[k] for k in ("notebook", "generated_utc", "fork_git_head")},
                 indent=2))
print(f"batch: {len(STEP4_CASES)} runs; totals {meta['design']['run_arithmetic']}")
print(f"full metadata: {EXPORTS / 'step4_metadata.json'}")


{
  "notebook": "step4_case_export.ipynb",
  "generated_utc": "2026-08-13T16:08:24+00:00",
  "fork_git_head": "26420712de33eddde7b9ff3cef3e43ca180dfcb0"
}
batch: 120 runs; totals {'step3_spent': 25, 'step4_batch': 120, 'total': 145, 'ceiling': '145 (amended 2026-08-13 from 163 on cancellation of the int arm; 163 ratified 2026-08-12; previously 126 -> 133 on 2026-08-10)'}
full metadata: C:\Users\ethan\code\research\ReEDS-nuclear-learning\z-ethan\mc\exports\step4\step4_metadata.json


## Closing note

The Step 4 batch is **one case file, 120 runs**: `cases_nuclearlearning_step4.csv`
(launch: `python runreeds.py -b <batch> -c nuclearlearning_step4`). Email notes for NREL:

- `demand_EER2025_100by2050.h5` (the `demhi` arm) is fetched automatically from Zenodo via
  `inputs/remote_files.csv` on first use — no action needed, just a heads-up.
- Everything else reuses input files already shipped with the Step 3 batch; the 12
  `large100_{sched}_{p05|p95}` cases carry their own new input files (included).

All 120 runs use the sequential solver (the 18-column intertemporal arm was cancelled
2026-08-13 — NREL: the intertemporal solver has not worked for years). Analysis-side
reminder recorded in the metadata: the issue-7 layered robustness invariant (bridge shape /
ranking / level).
